[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bytebyteai/ai-eng-projects/blob/main/project_1/lm_playground_solution.ipynb)

In [ ]:
import torch, transformers, tiktoken
print("torch", torch.__version__, "| transformers", transformers.__version__)

ModuleNotFoundError: No module named 'torch'

# 1 - Tokenization

A neural network can’t digest raw text. They need **numbers**. Tokenization is the process of converting text into IDs. In this section, you'll learn how tokenization is implemented in practice.

Tokenization methods generally fall into three categories:
1. Word-level
2. Character-level
3. Subword-level

### 1.1 - Word‑level tokenization

Split text on whitespace and store each **word** as a token.

In [1]:
# 1. Tiny corpus
corpus = [
    "The quick brown fox jumps over the lazy dog",
    "Tokenization converts text to numbers",
    "Large language models predict the next token"
]

# 2. Build the vocabulary
PAD, UNK = "[PAD]", "[UNK]"
words = set()
for doc in corpus:
    words.update(doc.lower().split())

vocab = [PAD, UNK] + sorted(words)
word2id = {w: i for i, w in enumerate(vocab)}
id2word = {i: w for w, i in word2id.items()}

print(f"Vocabulary size: {len(vocab)} words")
print("First 15 vocab entries:", vocab[:15])

# 3. Encode / decode
def encode(text):
    return [word2id.get(w, word2id[UNK]) for w in text.lower().split()]

def decode(ids):
    return " ".join(id2word[i] for i in ids if i != word2id[PAD])

# 4. Demo
sample = "The brown unicorn jumps"
ids = encode(sample)
recovered = decode(ids)

print("\nInput text :", sample)
print("Token IDs  :", ids)
print("Decoded    :", recovered)


Vocabulary size: 21 words
First 15 vocab entries: ['[PAD]', '[UNK]', 'brown', 'converts', 'dog', 'fox', 'jumps', 'language', 'large', 'lazy', 'models', 'next', 'numbers', 'over', 'predict']

Input text : The brown unicorn jumps
Token IDs  : [17, 2, 1, 6]
Decoded    : the brown [UNK] jumps


Word-level tokenization has two major limitations:
1. Large vocabulary size
2. Out-of-vocabulary (OOV) issue

### 1.2 - Character‑level tokenization

Every single character (including spaces and emojis) gets its own ID. This guarantees zero out‑of‑vocabulary issues but very long sequences.

In [ ]:
# 1. Build a fixed vocabulary
import string

letters = list(string.ascii_lowercase + string.ascii_uppercase)  # a–z + A–Z
special = ["[PAD]", "[UNK]"]  # padding + unknown
vocab = special + letters

char2id = {ch: idx for idx, ch in enumerate(vocab)}
id2char = {idx: ch for ch, idx in char2id.items()}

print(f"Vocabulary size: {len(vocab)} (52 letters + 2 specials)")

# 2. Encode / decode
def encode(text):
    """Convert text → list of IDs (unknown chars → [UNK])."""
    unk_id = char2id["[UNK]"]
    return [char2id.get(ch, unk_id) for ch in text]

def decode(ids):
    """Convert list of IDs."""
    return "".join(id2char[i] for i in ids if i != char2id["[PAD]"])

# 3. Demo
sample = "Hello"
ids = encode(sample)
recovered = decode(ids)

print("\nInput text :", sample)
print("Token IDs  :", ids)
print("Decoded    :", recovered)


### 1.3 - Subword‑level tokenization

Sub-word methods such as `Byte-Pair Encoding (BPE)`, `WordPiece`, and `SentencePiece` **learn** the most common character and gorup them into new tokens. For example, the word `unbelievable` might turn into three tokens: `["un", "believ", "able"]`. This approach strikes a balance between word-level and character-level methods and fix their limitations.

For example, `BPE` algorithm forms the vocabulary using the following steps:
1. **Start with bytes** → every character is its own token.  
2. **Count all adjacent pairs** in a huge corpus.  
3. **Merge the most frequent pair** into a new token.  
   *Repeat steps 2-3* until you hit the target vocab size (e.g., 50 k).

Let's see `BPE` in practice.

In [ ]:
# 1. Load a pretrained BPE tokenizer (GPT-2 uses BPE)
from transformers import AutoTokenizer

bpe_tok = AutoTokenizer.from_pretrained("gpt2")

print("Vocab size:", bpe_tok.vocab_size)
print("Special tokens:", bpe_tok.all_special_tokens)

# 2. Encode / decode
def encode(text):
    return bpe_tok.encode(text)

def decode(ids):
    return bpe_tok.decode(ids)

# 3. Demo
sample = "Unbelievable tokenization powers! 🚀"
ids = encode(sample)
recovered = decode(ids)

print("\nInput text :", sample)
print("Token IDs  :", ids)
print("Tokens     :", bpe_tok.convert_ids_to_tokens(ids))
print("Decoded    :", recovered)


### 1.4 - TikToken

`tiktoken` is a production-ready library which offers high‑speed tokenization used by OpenAI models.  
Let's compare the older **gpt2** encoding with the newer **cl100k_base** used in GPT‑4.

In [ ]:
import tiktoken

encodings = [
    ("gpt2", tiktoken.get_encoding("gpt2")),
    ("cl100k_base", tiktoken.get_encoding("cl100k_base")),
]

sentence = "The 🌟 star-player scored 40 points!"

for name, enc in encodings:
    print(f"\n=== {name} ===")
    print("Vocabulary size:", enc.n_vocab)

    # Encode the sample sentence
    ids = enc.encode(sentence)
    tokens = [enc.decode([i]) for i in ids]
    print(f"Sentence splits into {len(ids)} tokens:")
    print(list(zip(tokens, ids)))

    # Show a few arbitrary token→ID examples from the vocab
    some_ids = [0, 1, 2, 198, 50256]
    print("Sample tokens from the vocabulary:")
    print([(enc.decode([i]), i) for i in some_ids])


Experiment: try new sentences, emojis, code snippets, or other languages. If you are interested, try implementing the BPE algorithm yourself.

### 1.5 - Key Takeaways

* **Word‑level**: simple but brittle (OOV problems).  
* **Character‑level**: robust but produces long sequences.  
* **BPE / Byte‑Level BPE**: middle ground used by most LLMs.  
* **tiktoken**: shows how production models tokenize with pre‑trained sub‑word vocabularies.